In [1]:
import torch
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

/mnt/f/whisper-finetune/env_whisper-finetune/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("changelinglab/easycall-dysarthria")

print(dataset)
print(dataset["train"][0])

Found cached dataset parquet (/home/hillol/.cache/huggingface/datasets/changelinglab___parquet/changelinglab--easycall-dysarthria-40615a80a37ddc0a/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)
100%|██████████| 3/3 [00:00<00:00, 40.01it/s]


DatasetDict({
    test: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 5213
    })
    validation: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 4272
    })
    train: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 11901
    })
})
{'audio': {'path': 'f10_02_Scendi.wav', 'array': array([0., 0., 0., ..., 0., 0., 0.]), 'sampling_rate': 8000}, 'filename': 'f10_02_Scendi.wav', 'speaker': 'f10', 'text': 'scendi', 'dysarthria_severity': '3'}


In [3]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [4]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="italian",
    task="transcribe"
)

In [5]:
from datasets import disable_caching
disable_caching()

In [6]:
def normalize(text):
    return text.lower().strip()

def prepare(batch):
    audio_arrays = [x["array"] for x in batch["audio"]]

    batch["input_features"] = processor.feature_extractor(
        audio_arrays,
        sampling_rate=16000,
    ).input_features

    batch["labels"] = processor.tokenizer(
        [normalize(t) for t in batch["text"]],
        padding=True,
        truncation=True
    ).input_ids

    return batch

In [7]:
dataset = dataset.with_transform(prepare)

In [8]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="italian",
    task="transcribe"
)

In [9]:
for name, param in model.model.encoder.named_parameters():
    if "layers.10" in name or "layers.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [18]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [normalize(s) for s in pred_str]
    label_str = [normalize(s) for s in label_str]

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [20]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-dysarthria-it",

    per_device_train_batch_size=6,
    gradient_accumulation_steps=2,

    learning_rate=3e-6,
    warmup_steps=100,

    num_train_epochs=10,

    fp16=True,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=64,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    dataloader_num_workers=0,
    remove_unused_columns=False,

    label_smoothing_factor=0.1,
    
)

In [12]:
import setuptools
import distutils.version

In [ ]:
from dataclasses import dataclass
from typing import Any
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    model: Any

    def __call__(self, features):

        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"]

        # manual shift 
        decoder_input_ids = labels.clone()
        decoder_input_ids[:, 1:] = labels[:, :-1]
        decoder_input_ids[:, 0] = self.processor.tokenizer.bos_token_id

        # mask padding
        labels = labels.masked_fill(
            labels == self.processor.tokenizer.pad_token_id,
            -100
        )

        batch["labels"] = labels
        batch["decoder_input_ids"] = decoder_input_ids

        return batch

In [14]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    model=model
)

In [15]:
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Wer
1,1.455500,5.843303,8.471375
2,1.445600,3.992048,8.867618
3,1.437800,5.410616,11.359279
